In [1]:
# =========================================================
# RPDA8412 RESEARCH PROJECT
# NOTEBOOK 03: DATA ALIGNMENT
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FINAL_DIR = PROJECT_ROOT / "data" / "final"

FINAL_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Processed files:")

for file in sorted(PROCESSED_DIR.iterdir()):
    print("-", file.name)

Project: C:\Users\ibrah\RPDA8412 Research Project
Processed files:
- epu_daily_clean.csv
- epu_monthly_clean.csv
- gpr_daily_clean.csv
- gpr_monthly_clean.csv
- sp500_daily_clean.csv


In [2]:
# =========================================================
# LOAD CLEANED DATASETS
# =========================================================

sp500 = pd.read_csv(
    PROCESSED_DIR / "sp500_daily_clean.csv",
    parse_dates=["Date"]
)

gpr_daily = pd.read_csv(
    PROCESSED_DIR / "gpr_daily_clean.csv",
    parse_dates=["Date"]
)

epu_daily = pd.read_csv(
    PROCESSED_DIR / "epu_daily_clean.csv",
    parse_dates=["Date"]
)

gpr_monthly = pd.read_csv(
    PROCESSED_DIR / "gpr_monthly_clean.csv",
    parse_dates=["Date"]
)

epu_monthly = pd.read_csv(
    PROCESSED_DIR / "epu_monthly_clean.csv",
    parse_dates=["Date"]
)

for name, df in [
    ("S&P 500", sp500),
    ("GPR Daily", gpr_daily),
    ("EPU Daily", epu_daily),
    ("GPR Monthly", gpr_monthly),
    ("EPU Monthly", epu_monthly)
]:
    print(
        f"{name}: {len(df)} observations | "
        f"{df['Date'].min().date()} to "
        f"{df['Date'].max().date()}"
    )

S&P 500: 4024 observations | 2010-01-04 to 2025-12-31
GPR Daily: 5844 observations | 2010-01-01 to 2025-12-31
EPU Daily: 5844 observations | 2010-01-01 to 2025-12-31
GPR Monthly: 192 observations | 2010-01-01 to 2025-12-01
EPU Monthly: 192 observations | 2010-01-01 to 2025-12-01


In [3]:
# =========================================================
# DAILY DATE ALIGNMENT
# Descriptive analysis only
# =========================================================

daily_aligned = (
    sp500
    .merge(
        gpr_daily,
        on="Date",
        how="left",
        validate="one_to_one"
    )
    .merge(
        epu_daily,
        on="Date",
        how="left",
        validate="one_to_one"
    )
    .sort_values("Date")
    .reset_index(drop=True)
)

print("Aligned dataset shape:", daily_aligned.shape)

print("\nColumns:")
print(daily_aligned.columns.tolist())

print("\nMissing values:")
print(daily_aligned.isna().sum())

print("\nFirst five observations:")
display(daily_aligned.head())

print("\nLast five observations:")
display(daily_aligned.tail())

Aligned dataset shape: (4024, 11)

Columns:
['Date', 'Adj_Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'GPR', 'GPR_Acts', 'GPR_Threats', 'EPU']

Missing values:
Date           0
Adj_Close      0
Close          0
High           0
Low            0
Open           0
Volume         0
GPR            0
GPR_Acts       0
GPR_Threats    0
EPU            0
dtype: int64

First five observations:


,Date,Adj_Close,Close,High,Low,Open,Volume,GPR,GPR_Acts,GPR_Threats,EPU
0,2010-01-04,1132.989990,1132.989990,1133.869995,1116.560059,1116.560059,3991400000,177.117065,190.888504,214.946838,150.35
1,2010-01-05,1136.520020,1136.520020,1136.630005,1129.660034,1132.660034,2491020000,122.525620,148.323151,123.716156,151.16
2,2010-01-06,1137.140015,1137.140015,1139.189941,1133.949951,1135.709961,4972660000,161.588669,177.200409,161.239243,95.95
3,2010-01-07,1141.689941,1141.689941,1142.459961,1131.319946,1136.270020,5270680000,102.262291,116.511452,91.108170,92.20
4,2010-01-08,1144.979980,1144.979980,1145.390015,1136.219971,1140.520020,4389590000,110.296379,117.287331,110.057869,141.92



Last five observations:


,Date,Adj_Close,Close,High,Low,Open,Volume,GPR,GPR_Acts,GPR_Threats,EPU
4019,2025-12-24,6932.049805,6932.049805,6937.319824,6904.910156,6904.910156,1798270000,134.293274,89.253281,186.115112,204.03
4020,2025-12-26,6929.939941,6929.939941,6945.770020,6921.600098,6936.020020,2586550000,144.024689,159.534836,186.294937,513.83
4021,2025-12-29,6905.740234,6905.740234,6920.209961,6888.759766,6903.600098,3541750000,62.946659,47.064674,65.427643,445.72
4022,2025-12-30,6896.240234,6896.240234,6913.250000,6893.470215,6900.439941,3309930000,218.892136,160.026810,266.956329,305.80
4023,2025-12-31,6845.500000,6845.500000,6901.419922,6844.549805,6898.819824,3261830000,161.760208,140.228073,210.535324,307.63


In [4]:
# =========================================================
# PROVISIONAL LAGGED GPR/EPU PREDICTORS
# =========================================================

# Construct calendar-day uncertainty dataset
uncertainty_daily = (
    gpr_daily[["Date", "GPR"]]
    .merge(
        epu_daily[["Date", "EPU"]],
        on="Date",
        how="inner",
        validate="one_to_one"
    )
    .sort_values("Date")
    .reset_index(drop=True)
)

# Values from the previous calendar day
uncertainty_daily["GPR_lag1"] = (
    uncertainty_daily["GPR"].shift(1)
)

uncertainty_daily["EPU_lag1"] = (
    uncertainty_daily["EPU"].shift(1)
)

# Merge onto S&P 500 trading calendar
daily_forecast_base = (
    sp500
    .merge(
        uncertainty_daily[
            ["Date", "GPR_lag1", "EPU_lag1"]
        ],
        on="Date",
        how="left",
        validate="one_to_one"
    )
    .sort_values("Date")
    .reset_index(drop=True)
)

print("Forecast base shape:", daily_forecast_base.shape)

print("\nMissing values:")
print(daily_forecast_base.isna().sum())

display(daily_forecast_base.head())

Forecast base shape: (4024, 9)

Missing values:
Date         0
Adj_Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
GPR_lag1     0
EPU_lag1     0
dtype: int64


,Date,Adj_Close,Close,High,Low,Open,Volume,GPR_lag1,EPU_lag1
0,2010-01-04,1132.989990,1132.989990,1133.869995,1116.560059,1116.560059,3991400000,91.471298,216.95
1,2010-01-05,1136.520020,1136.520020,1136.630005,1129.660034,1132.660034,2491020000,177.117065,150.35
2,2010-01-06,1137.140015,1137.140015,1139.189941,1133.949951,1135.709961,4972660000,122.525620,151.16
3,2010-01-07,1141.689941,1141.689941,1142.459961,1131.319946,1136.270020,5270680000,161.588669,95.95
4,2010-01-08,1144.979980,1144.979980,1145.390015,1136.219971,1140.520020,4389590000,102.262291,92.20


In [5]:
# =========================================================
# ALIGNMENT VALIDATION
# =========================================================

assert len(daily_aligned) == len(sp500)
assert len(daily_forecast_base) == len(sp500)

assert daily_aligned["Date"].is_unique
assert daily_forecast_base["Date"].is_unique

assert daily_aligned["Date"].is_monotonic_increasing
assert daily_forecast_base["Date"].is_monotonic_increasing

assert daily_aligned[["GPR", "EPU"]].notna().all().all()

assert daily_forecast_base[
    ["GPR_lag1", "EPU_lag1"]
].notna().all().all()

print("ALL DAILY ALIGNMENT CHECKS PASSED!")
print("Trading-day observations:", len(sp500))

ALL DAILY ALIGNMENT CHECKS PASSED!
Trading-day observations: 4024


In [6]:
# =========================================================
# SAVE ALIGNED DATASETS
# =========================================================

daily_aligned.to_csv(
    FINAL_DIR / "daily_aligned_descriptive.csv",
    index=False
)

daily_forecast_base.to_csv(
    FINAL_DIR / "daily_forecast_base_provisional.csv",
    index=False
)

print("Saved descriptive dataset.")
print("Saved provisional forecasting dataset.")

Saved descriptive dataset.
Saved provisional forecasting dataset.
